# Layer 1 — Quality Tests

Validates that `data/processed_kpi_features.csv` and `data/kpi_anomaly_detection.db` produced by `scripts/1.2_1.3_ingest_and_engineer.py` meet all structural and statistical expectations before Layer 2 runs.

| # | Test | What it checks |
|---|------|----------------|
| 1 | Shape | 731 rows × 183 columns |
| 2 | Column naming | 25 columns per feature suffix |
| 3 | KPI tier coverage | All 12 tiered KPIs present with engineered features |
| 4 | Rolling mean never NaN | min_periods=1 guarantees full coverage |
| 5 | NaN warm-up pattern | lag_1, wow_change, mom_change warm-up rows are NaN |
| 6 | Z-score sanity | Non-anomaly rows rarely exceed \|z\| > 4 |
| 7 | SQLite parity | DB table matches CSV shape |
| 8 | Date continuity | Exactly 731 consecutive days, no gaps |

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import os

# Ensure paths resolve from project root regardless of where notebook is launched
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'scripts':
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

CSV_PATH = PROJECT_ROOT / 'data' / 'processed_kpi_features.csv'
DB_PATH  = PROJECT_ROOT / 'data' / 'kpi_anomaly_detection.db'

df = pd.read_csv(CSV_PATH)

print(f'Loaded: {CSV_PATH.relative_to(PROJECT_ROOT)}')
print(f'Shape:  {df.shape}')
print(f'Dates:  {df["date"].iloc[0]}  →  {df["date"].iloc[-1]}')

Loaded: data\processed_kpi_features.csv
Shape:  (731, 183)
Dates:  2024-01-01  →  2025-12-31


---
## Test 1 — Shape
Expected: `PASS  shape: (731, 183)`

In [2]:
assert df.shape[0] == 731, f'Expected 731 rows, got {df.shape[0]}'
assert df.shape[1] == 183, f'Expected 183 cols (33 raw + 25 KPIs x 6), got {df.shape[1]}'
print(f'PASS  shape: {df.shape}')

PASS  shape: (731, 183)


---
## Test 2 — Column Naming Convention
Every engineered column must follow `{kpi}_{feature}` and each of the 6 suffixes must appear exactly 25 times (once per KPI).

In [3]:
suffixes = ['_rolling_mean', '_rolling_std', '_z_score',
            '_wow_change',   '_mom_change',  '_lag_1']

for sfx in suffixes:
    count = sum(c.endswith(sfx) for c in df.columns)
    assert count == 25, f'{sfx}: expected 25, got {count}'
    print(f'PASS  {sfx}: {count} columns')

PASS  _rolling_mean: 25 columns
PASS  _rolling_std: 25 columns
PASS  _z_score: 25 columns
PASS  _wow_change: 25 columns
PASS  _mom_change: 25 columns
PASS  _lag_1: 25 columns


---
## Test 3 — KPI Tier Coverage
All 12 tiered KPIs must be present as both a raw column and as engineered features.

In [4]:
TIER_1 = ['total_revenue_usd', 'n_orders', 'avg_roas', 'conversion_rate']
TIER_2 = ['return_rate', 'n_stockouts', 'avg_order_value_usd', 'bounce_rate']
TIER_3 = ['total_clicks', 'sessions', 'inventory_health', 'avg_discount_pct']

for kpi in TIER_1 + TIER_2 + TIER_3:
    assert kpi in df.columns,                   f'Missing raw column: {kpi}'
    assert f'{kpi}_z_score' in df.columns,       f'Missing z_score for: {kpi}'
    assert f'{kpi}_rolling_mean' in df.columns,  f'Missing rolling_mean for: {kpi}'
    print(f'PASS  {kpi}')

PASS  total_revenue_usd
PASS  n_orders
PASS  avg_roas
PASS  conversion_rate
PASS  return_rate
PASS  n_stockouts
PASS  avg_order_value_usd
PASS  bounce_rate
PASS  total_clicks
PASS  sessions
PASS  inventory_health
PASS  avg_discount_pct


---
## Test 4 — Rolling Mean Never NaN
`_rolling_mean` uses `min_periods=1`, so every row including row 0 must have a value.

In [5]:
rolling_mean_cols = [c for c in df.columns if c.endswith('_rolling_mean')]

for col in rolling_mean_cols:
    n_null = df[col].isna().sum()
    assert n_null == 0, f'{col}: {n_null} unexpected NaN values'

print(f'PASS  rolling_mean: 0 NaN across all {len(rolling_mean_cols)} columns')

PASS  rolling_mean: 0 NaN across all 25 columns


---
## Test 5 — Expected NaN Pattern (warm-up rows)
Lag and window features must be NaN during their warm-up period and fully populated thereafter.

In [6]:
# lag_1: only row 0 is NaN
lag_cols = [c for c in df.columns if c.endswith('_lag_1')]
for col in lag_cols:
    assert df[col].isna().sum() == 1, f'{col}: expected exactly 1 NaN (row 0)'
print(f'PASS  lag_1: exactly 1 NaN (row 0) across all {len(lag_cols)} columns')

# wow_change: rows 0-6 are NaN (7 rows)
wow_cols = [c for c in df.columns if c.endswith('_wow_change')]
for col in wow_cols:
    warm_up_nulls = df[col].iloc[:7].isna().sum()
    assert warm_up_nulls == 7, f'{col}: expected 7 NaN in warm-up, got {warm_up_nulls}'
print(f'PASS  wow_change: exactly 7 NaN in warm-up rows across all {len(wow_cols)} columns')

# mom_change: rows 0-29 are NaN (30 rows)
mom_cols = [c for c in df.columns if c.endswith('_mom_change')]
for col in mom_cols:
    warm_up_nulls = df[col].iloc[:30].isna().sum()
    assert warm_up_nulls == 30, f'{col}: expected 30 NaN in warm-up, got {warm_up_nulls}'
print(f'PASS  mom_change: exactly 30 NaN in warm-up rows across all {len(mom_cols)} columns')

print()
print('PASS  NaN warm-up pattern correct for lag_1, wow_change, mom_change')

PASS  lag_1: exactly 1 NaN (row 0) across all 25 columns
PASS  wow_change: exactly 7 NaN in warm-up rows across all 25 columns
PASS  mom_change: exactly 30 NaN in warm-up rows across all 25 columns

PASS  NaN warm-up pattern correct for lag_1, wow_change, mom_change


---
## Test 6 — Z-Score Sanity (non-anomaly days)
On non-anomaly days the Z-score should rarely exceed ±4. A high proportion of extreme Z-scores indicates a data or calculation error.

In [7]:
z_cols       = [c for c in df.columns if c.endswith('_z_score')]
non_anomaly  = df[df['anomaly_flag'] == 0]

for col in z_cols:
    extreme = non_anomaly[col].abs().gt(4).sum()
    total   = non_anomaly[col].notna().sum()
    pct     = extreme / total if total > 0 else 0
    assert pct < 0.02, f'{col}: {pct:.1%} of non-anomaly rows have |z| > 4 (threshold: 2%)'

print(f'PASS  Z-scores within expected range on non-anomaly days')
print(f'      Checked {len(z_cols)} z_score columns across {len(non_anomaly)} non-anomaly rows')

PASS  Z-scores within expected range on non-anomaly days
      Checked 25 z_score columns across 711 non-anomaly rows


---
## Test 7 — SQLite Parity
The SQLite table must contain the same number of rows and columns as the CSV.

In [8]:
conn  = sqlite3.connect(DB_PATH)
db_df = pd.read_sql('SELECT * FROM processed_kpis', conn)
conn.close()

assert db_df.shape == df.shape, (
    f'SQLite shape {db_df.shape} does not match CSV shape {df.shape}'
)
print(f'PASS  SQLite table matches CSV: {db_df.shape}')

PASS  SQLite table matches CSV: (731, 183)


---
## Test 8 — Date Continuity
The processed dataset must have exactly 731 consecutive calendar days with no gaps.

In [9]:
dates = pd.to_datetime(df['date'])

assert dates.min().date().isoformat() == '2024-01-01', f'Unexpected start date: {dates.min().date()}'
assert dates.max().date().isoformat() == '2025-12-31', f'Unexpected end date:   {dates.max().date()}'

gaps = dates.sort_values().diff().iloc[1:]
assert (gaps == pd.Timedelta('1 day')).all(), 'Date gaps detected — missing rows in source data'

print(f'PASS  Date range {dates.min().date()} to {dates.max().date()}, no gaps')
print(f'      Total days: {len(dates)}')

PASS  Date range 2024-01-01 to 2025-12-31, no gaps
      Total days: 731


---
## Summary

In [10]:
tests = [
    ('Test 1', 'Shape'),
    ('Test 2', 'Column Naming Convention'),
    ('Test 3', 'KPI Tier Coverage'),
    ('Test 4', 'Rolling Mean Never NaN'),
    ('Test 5', 'Expected NaN Pattern'),
    ('Test 6', 'Z-Score Sanity'),
    ('Test 7', 'SQLite Parity'),
    ('Test 8', 'Date Continuity'),
]

print('=' * 50)
print('Layer 1 Quality Test Results')
print('=' * 50)
for num, name in tests:
    print(f'  PASS  {num} — {name}')
print('=' * 50)
print(f'  All 8 tests passed — Layer 1 output is valid.')
print(f'  processed_kpi_features.csv is ready for Layer 2.')
print('=' * 50)

Layer 1 Quality Test Results
  PASS  Test 1 — Shape
  PASS  Test 2 — Column Naming Convention
  PASS  Test 3 — KPI Tier Coverage
  PASS  Test 4 — Rolling Mean Never NaN
  PASS  Test 5 — Expected NaN Pattern
  PASS  Test 6 — Z-Score Sanity
  PASS  Test 7 — SQLite Parity
  PASS  Test 8 — Date Continuity
  All 8 tests passed — Layer 1 output is valid.
  processed_kpi_features.csv is ready for Layer 2.
